https://milvus.io/docs/es/milvus_lite.md

# Configuración global


# Conexión a Milvus Lite

 COLECCIÓN 1  "conocimiento"

In [8]:

from pathlib import Path

BASE_DIR = Path.cwd() / "pmai-model-vision-language"
MILVUS_DB_PATH    = str(BASE_DIR / "vectordb" / "androide_milvus.db")
EMBED_MODEL_NAME  = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM         = 384
COL_CONOCIMIENTO  = "conocimiento"
UMBRAL_CONOCIMIENTO = 0.85

In [9]:
from pymilvus import MilvusClient
client = MilvusClient("./androide_milvus.db")

print("Colecciones existentes:", client.list_collections())



Colecciones existentes: []


primer schema de la coleccion a


In [12]:
from pymilvus import DataType, MilvusClient
import numpy as np

schema = client.create_schema(
    auto_id=True,                
    enable_dynamic_field=False,  
)


schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="pregunta", datatype=DataType.VARCHAR, max_length=1000)
schema.add_field(field_name="respuesta", datatype=DataType.VARCHAR, max_length=3000)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=EMBED_DIM)




{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'pregunta', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 1000}}, {'name': 'respuesta', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 3000}}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}

I0607 20:21:05.050313 1037929 chttp2_transport.cc:1369] ipv4:127.0.0.1:60870: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0607 20:21:05.050538 1037929 chttp2_transport.cc:1401] ipv4:127.0.0.1:60870: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [13]:
client.create_collection(
    collection_name=COL_CONOCIMIENTO,   # nombre: "conocimiento"
    schema=schema,                       # el plano que definiste
)

In [ ]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="HNSW",          
    metric_type="COSINE",       
    params={"M": 16, "efConstruction": 200},   
)

client.create_index(
    collection_name=COL_CONOCIMIENTO,
    index_params=index_params,
)



2026-06-07 20:42:02,623 [ERROR][_log_rpc_error]: RPC error: [create_index], <MilvusException: (code=35, message=index already exists for field 'vector'; call drop_index first)>, <elapsed:1.6ms>
Traceback:
Traceback (most recent call last):
  File "/Users/anpalop/Desktop/pmai-model-vision-language/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 518, in handler
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anpalop/Desktop/pmai-model-vision-language/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 565, in handler
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anpalop/Desktop/pmai-model-vision-language/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 456, in handler
    raise e from e
  File "/Users/anpalop/Desktop/pmai-model-vision-language/.venv/lib/python3.12/site-packages/pymilvus/decorators.py", line 419, in handler
    return func(*args, **kwargs)
        

MilvusException: <MilvusException: (code=35, message=index already exists for field 'vector'; call drop_index first)>

In [ ]:
client.load_collection(collection_name=COL_CONOCIMIENTO)

Estado: {'state': <LoadState: Loaded>}


I0607 20:44:33.885769 1037931 chttp2_transport.cc:1369] ipv4:127.0.0.1:60870: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0607 20:44:33.885882 1037931 chttp2_transport.cc:1401] ipv4:127.0.0.1:60870: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 40000ms


In [ ]:
from sentence_transformers import SentenceTransformer
modelo_embed = SentenceTransformer(EMBED_MODEL_NAME)

/Users/anpalop/Desktop/pmai-model-vision-language/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0607 20:47:55.103283 1023379 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0607 20:47:55.110138 1110803 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(93, generation: 1)
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 22850.67it/s]


Tipo: <class 'numpy.ndarray'>
Dimensión: 384
Primeros 5 números: [ 0.20666677  0.12004524  0.29755005  0.39185345 -0.2293803 ]
